## Setup Unsloth for 4-bit Quantization

First, we need to install the `unsloth` library and its dependencies. This process includes installing `xformers` and `bitsandbytes` for efficient 4-bit quantization.

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Put this at the very top of your notebook, before any imports

In [3]:
# Install bitsandbytes separately from PyPI
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00


In [4]:
# Install Unsloth and fixed versions of dependencies to avoid building from source
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.30" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4a67_ptq/unsloth_22e7e819dca342079847851f9672554c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4a67_ptq/unsloth_22e7e819dca342079847851f9672554c
  Resolved https://github.com/unslothai/unsloth.git to commit ebf28e7e07212bd41756e9b3390073021a3ba1a0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.6.1-py3-none-any.whl size=34849948 sha256=c4c1a260fdaedceee86865704c029d04a186147f043f04d37df64569e2fb8b9f
  Stored in directory: /tmp/pip-ephem-wheel-cache-hwg0hq8w/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 28.6 MB/s eta 0:00:00


Next, let's import the necessary modules from `unsloth` and `transformers`.

In [5]:
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Su

In [6]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


Now, we'll load a pre-trained model and its tokenizer using Unsloth's `FastLanguageModel`. This will automatically apply 4-bit quantization and prepare the model for fine-tuning or inference.

In [7]:
max_seq_length = 1024 # Reduced from 2048 to prevent OOM
dtype = None # None for auto detection.
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

### 1. Add LoRA Adapters
We add LoRA adapters to the model so we only need to train 1% to 10% of all parameters.

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### 2. Prepare Dataset
Replace `'YOUR_DATASET_PATH'` with your actual Hugging Face dataset ID.

In [9]:
from datasets import load_dataset

dataset_name = "autummata/kernelbook-verified"
# Load and split: 80% train, 20% test (validation)
raw_dataset = load_dataset(dataset_name, split = "train").train_test_split(test_size=0.2, seed=3407)

def formatting_prompts_func(examples):
    python_snippets = examples["python_code"]
    triton_kernels  = examples["triton_code"]
    # Gracefully handle the column if it's missing or named slightly differently
    triton_is_faster = examples.get("triton_is_faster", ["Unknown"] * len(python_snippets))
    texts = []
    for python, triton, is_faster in zip(python_snippets, triton_kernels, triton_is_faster):
        # Injecting the triton_is_faster flag into the system prompt
        text = f"<|im_start|>system\nYou are an expert GPU programmer specializing in Triton kernels. Convert the following PyTorch code into an optimized Triton kernel using tl.load/tl.store with masking, power-of-2 block sizes tuned for H100/A100, and kernel fusion where possible. Include a Python wrapper that infers the grid and a torch.allclose correctness test. Output only valid Python, no markdown fences, no prose.<|im_end|>\n"
        text += f"<|im_start|>user\n{python}<|im_end|>\n"
        text += f"<|im_start|>assistant\n{triton}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

train_dataset = raw_dataset["train"].map(formatting_prompts_func, batched = True)
eval_dataset = raw_dataset["test"].map(formatting_prompts_func, batched = True)


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

kernelbook_benchmarked.parquet:   0%|          | 0.00/62.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14429 [00:00<?, ? examples/s]

Map:   0%|          | 0/11543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2886 [00:00<?, ? examples/s]

### 2.1 Refined Dataset Formatting
Since the dataset contains specific columns for Python and Triton code, we will format the prompts to instruct the model to perform the translation task.

### 3. Initialize Trainer
Using the `SFTTrainer` for supervised fine-tuning.

In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
      per_device_train_batch_size = 4,
      gradient_accumulation_steps = 2,
      warmup_steps = 15,
      max_steps = 500,
      num_train_epochs = 1,  # keep as backup
      learning_rate = 2e-4,
      lr_scheduler_type = "cosine",
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      logging_steps = 25,
      eval_strategy = "steps",
      eval_steps = 300,
      optim = "adamw_8bit",
      weight_decay = 0.01,
      gradient_checkpointing = False,
      load_best_model_at_end = False,
      save_strategy = "steps",
      save_steps = 300,
      save_total_limit = 1,
      max_seq_length = 1024,          # ← match model load
      packing = True,                # ← Unsloth skips it anyway, stop wasting overhead
      dataset_text_field = "text",    # ← add back now that packing=False
      dataset_num_proc = 4,
      seed = 3407,
      output_dir = "outputs",
    ),
)

Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/11543 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/2886 [00:00<?, ? examples/s]

### 4. Execute Training
This will start the fine-tuning process. We will monitor the loss to ensure the model is learning the mapping from Python to Triton.

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,543 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 29,097,984 of 9,438,911,728 (0.31% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release alo

Step,Training Loss,Validation Loss


In [ ]:
import torch
from unsloth import FastLanguageModel

# Switch to inference mode
FastLanguageModel.for_inference(model)

# Define the PyTorch code you want to convert
pytorch_code = """
def mul(a, b):
    return a * b
"""

# 1. Define the message using the list-of-dicts format for content
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": """You are an expert GPU programmer specialized in writing Triton kernels.
Given a PyTorch function, your task is to write an equivalent Triton kernel using the @triton.jit decorator.
Only generate python code that can be directly executed. (No other formats allowed).

═══════════════════════════════════════════════════════
CORRECTNESS RULES
═══════════════════════════════════════════════════════
- Always use @triton.jit as the decorator
- Use tl.program_id(axis=N) to identify the current program, where N depends on the launch grid dimensions
- Use tl.load and tl.store for pointer-based memory access
- Parameters that are compile-time constants must be annotated with tl.constexpr
- Never use Python built-ins like len() directly on tensors
- Never use torch or numpy inside a kernel
- Python's range() is valid for iterating over integer ranges
- Always use masks when input size may not be a multiple of BLOCK_SIZE

═══════════════════════════════════════════════════════
PERFORMANCE RULES
═══════════════════════════════════════════════════════
- Process data in tiles using BLOCK_SIZE as a tl.constexpr — always a power of 2
- Access memory in contiguous, strided patterns to maximize bandwidth
- Fuse multiple operations into a single kernel to avoid multiple memory passes
- All block sizes and tile dimensions must be tl.constexpr

- Use tl.float32 accumulators for dot products, cast to lower precision at the end:
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    accumulator = tl.dot(a, b, accumulator)
    c = accumulator.to(tl.float16)

- For 2D kernels, reorder program IDs in groups to improve L2 cache reuse:
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

- Use num_stages in tl.range to enable software pipelining:
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        ...

- Subtract max before exp for numerical stability:
    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_output = numerator / denominator
"""}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": pytorch_code}]
    },
]

# 2. Apply the chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=2048,
    do_sample=False,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,
)

generated_tokens = outputs[0][inputs.shape[1]:]

triton_kernel = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(triton_kernel)


The model and tokenizer are now loaded and ready for use. You can proceed with fine-tuning or inference tasks.

### 6. Run Native TritonBench
Let's clone the official TritonBench repository and generate our own `predictions.jsonl` directly using the fine-tuned Unsloth model.

In [ ]:
%cd /content
!git clone https://github.com/thunlp/TritonBench.git
%cd TritonBench
!ls -la data/

In [ ]:
import json
import os
from tqdm import tqdm

data_path = "/content/TritonBench/data/TritonBench_T_simp_alpac_v1.json"
output_file = "/content/predictions.jsonl"

BATCH_SIZE = 166

if os.path.exists(data_path):
    with open(data_path, "r") as f:
        eval_data = json.load(f)

    print(f"Loaded {len(eval_data)} benchmark items.")

    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    results = []

    for i in tqdm(range(0, len(eval_data), BATCH_SIZE)):
      batch = eval_data[i : i + BATCH_SIZE]

      batch_prompts = []
      for item in batch:
          messages = [
              {"role": "system", "content": "You are an expert GPU programmer specializing in Triton kernels. Convert the following PyTorch code into an optimized Triton kernel output only valid Python, no markdown fences, no prose. You also cannot and should not use any type of pytorch in order to fulfill your goal, you must just output the Triton equivalent."},
              {"role": "user",   "content": item["instruction"]},
          ]
          prompt = tokenizer.apply_chat_template(  # original tokenizer for template
              messages,
              tokenize=False,
              add_generation_prompt=True,
          )
          batch_prompts.append(prompt)

      encoded = tokenizer(
            text=batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,        # up from 1024, fits longer instructions fully
            add_special_tokens=False,
      ).to("cuda")

      outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            max_new_tokens=1024,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
      )

      for j, (item, output) in enumerate(zip(batch, outputs)):
          input_len = encoded["input_ids"][j].shape[0]
          generated_tokens = output[input_len:]
          triton_code = tokenizer.decode(generated_tokens, skip_special_tokens=True)
          results.append({"instruction": item["instruction"], "predict": triton_code})
    with open(output_file, "w") as out_f:
        for res in results:
            out_f.write(json.dumps(res) + "\n")

    print(f"\nSaved {len(results)} predictions to {output_file}")


In [ ]:
import json
import ast
from tqdm import tqdm

predictions_path = "/content/predictions.jsonl"

stats = {
    "total": 0,
    "syntax_ok": 0,
    "syntax_fail": 0,
    "empty": 0,
    "contains_triton": 0,
}

syntax_errors = []

with open(predictions_path, "r") as f:
    predictions = [json.loads(line) for line in f]

for idx, item in enumerate(tqdm(predictions)):
    stats["total"] += 1

    code = item["predict"].strip()

    if not code:
        stats["empty"] += 1
        continue

    if "triton" in code.lower():
        stats["contains_triton"] += 1

    try:
        ast.parse(code)
        stats["syntax_ok"] += 1
    except Exception as e:
        stats["syntax_fail"] += 1
        syntax_errors.append(
            {
                "idx": idx,
                "error": str(e)
            }
        )

print("\n===== PRECHECK =====")
for k, v in stats.items():
    print(f"{k:20} {v}")

### 7. Clean and Prepare Kernels for TritonBench
We need to extract the actual Python code from the generated text and save them as individual `.py` files. We will use a regex to find python code blocks if they exist, or just try to clean the raw text.

In [ ]:
import json
import os
import re
import ast

predictions_path = "/content/predictions.jsonl"
output_dir = "/content/eval_llm_outputs/valid"
os.makedirs(output_dir, exist_ok=True)

with open("/content/TritonBench/data/TritonBench_T_simp_alpac_v1.json", "r") as f:
    gold_data = json.load(f)

with open(predictions_path, "r") as f:
    predictions = [json.loads(line) for line in f]

cleaned_stats = {"total": 0, "syntax_ok": 0, "syntax_fail": 0}

for i, pred in enumerate(predictions):
    raw_text = pred["predict"]

    text_no_think = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL)

    match = re.search(r"```python\n(.*?)```", text_no_think, flags=re.DOTALL)
    if match:
        code = match.group(1).strip()
    else:
        code = text_no_think.strip()
        code = re.sub(r"^=[0-9]+\s*", "", code)

    cleaned_stats["total"] += 1

    try:
        ast.parse(code)
        cleaned_stats["syntax_ok"] += 1

        # Only save if syntax is valid
        file_path = os.path.join(output_dir, f"{i}.py")
        with open(file_path, "w") as f:
            f.write(code)

    except SyntaxError:
        cleaned_stats["syntax_fail"] += 1
        print(f"Skipping prediction {i} due to syntax error.")

print("=== CLEANING RESULTS ===")
print(f"Total parsed: {cleaned_stats['total']}")
print(f"Valid Python Syntax (saved): {cleaned_stats['syntax_ok']}")
print(f"Invalid Syntax (skipped): {cleaned_stats['syntax_fail']}")
print(f"Saved to {output_dir}")

### 8. Run TritonBench Execution Accuracy
Now we run the execution evaluation. The script `1_exe_acc.py` uses multiprocessing to run the models simultaneously across the GPUs provided. We pass `0` to use the primary GPU (or `0,1,2,3` if you have multiple).

In [ ]:
# We need to patch the script slightly because it hardcodes the python interpreter and gold_folder path
import os

eval_script = "/content/TritonBench/EVAL/eval_T/1_exe_acc.py"
with open(eval_script, "r") as f:
    script_code = f.read()

# Patch interpreter to the colab environment python
script_code = script_code.replace('py_interpreter = "/home/lijianling/miniconda3/envs/LLM/bin/python"', 'py_interpreter = "python"')
# Patch gold folder to the simplified benchmark folder
script_code = script_code.replace('gold_folder = "data/TritonBench_T_v1/"', 'gold_folder = "/content/TritonBench/data/TritonBench_T_v1/"')

with open(eval_script, "w") as f:
    f.write(script_code)

print("Patched evaluation script.")

In [ ]:
!cd /content/TritonBench && python EVAL/eval_T/1_exe_acc.py --folder /content/eval_llm_outputs/valid --GPUs 0

In [ ]:
!cd /content/eval_llm_outputs/valid
!pwd
!find ../eval_llm_outputs/valid -type f -iname "*.py" | wc -l